# LLM Classification Finetuning — v0.4a Reward Zero-Shot

**Model:** `Skywork-Reward-Llama-3.1-8B-v0.2` — sequence classification reward model.  
**Strategy:** Score each (prompt, response) pair independently; convert score delta to
3-class probabilities (A / B / tie) with calibrated temperature + tie_weight. Swap-TTA
averages forward and swapped passes to cancel position bias.  
**Zero-shot:** no fine-tuning; calibration on fold 0 OOF only.

**Inputs:**
- `shelterw/skywork/Transformers/skywork-reward-llama-3.1-8b-v0.2/1` — reward model
- `llm-classification-finetuning` — competition data

**Push workflow:**
1. `zsh scripts/push_notebook.sh v0.4a-reward-zeroshot`
2. Stop the auto-run immediately in Kaggle UI
3. Accelerator → GPU T4 × 1 → Save Version → Save & Run All

In [ ]:
import subprocess, sys

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.46',
], check=True)

print('Dependencies installed')

In [ ]:
import glob
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.metrics import log_loss
from sklearn.model_selection import StratifiedKFold
from transformers import AutoModelForSequenceClassification, AutoTokenizer

warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f'GPU {i}: {torch.cuda.get_device_name(i)}  '
              f'{torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB')
else:
    print('CPU only')

# Calibration hyperparams — tuned on fold 0 OOF below
TEMPERATURE = 1.0   # sharpens/softens sigmoid over score delta
TIE_WEIGHT  = 0.3   # scales tie mass relative to A/B

MAX_SEQ_LEN = 2048
N_FOLDS     = 5
SEED        = 42
OUTPUT      = Path('/kaggle/working')

print('Imports OK')

In [ ]:
# Competition data
_data_candidates = [Path('/kaggle/input/llm-classification-finetuning')]
INPUT = next((p for p in _data_candidates if (p / 'test.csv').exists()), None)
if INPUT is None:
    _found = glob.glob('/kaggle/input/**/test.csv', recursive=True)
    INPUT = Path(_found[0]).parent if _found else None
assert INPUT is not None, 'Competition data not found'
print(f'Data : {INPUT}')

# Reward model — mounted from shelterw/skywork/Transformers/skywork-reward-llama-3.1-8b-v0.2/1
_model_configs = [
    p for p in glob.glob('/kaggle/input/**/config.json', recursive=True)
    if 'skywork' in p.lower() or 'reward' in p.lower() or 'llama' in p.lower()
]
MODEL_PATH = str(Path(_model_configs[0]).parent) if _model_configs else None
if MODEL_PATH is None:
    # Fallback: any model with a sequence classification config
    _all_configs = glob.glob('/kaggle/input/**/config.json', recursive=True)
    for cfg in _all_configs:
        import json
        try:
            with open(cfg) as f:
                c = json.load(f)
            if c.get('architectures') and 'SequenceClassification' in str(c['architectures']):
                MODEL_PATH = str(Path(cfg).parent)
                break
        except Exception:
            pass
assert MODEL_PATH is not None, (
    'Reward model not found — attach shelterw/skywork/Transformers/skywork-reward-llama-3.1-8b-v0.2/1'
)
print(f'Model: {MODEL_PATH}')

In [ ]:
print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)

print('Loading reward model (bf16, eager)...')
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.bfloat16,
    device_map='cuda:0',
    attn_implementation='eager',
    local_files_only=True,
)
model.eval()
print(f'Model ready — num_labels: {model.config.num_labels}')

In [ ]:
@torch.inference_mode()
def score_response(prompt: str, response: str) -> float:
    """Return reward scalar for a single (prompt, response) pair."""
    conv = [
        {'role': 'user',      'content': str(prompt   or '')},
        {'role': 'assistant', 'content': str(response or '')},
    ]
    text = tokenizer.apply_chat_template(conv, tokenize=False)
    enc = tokenizer(
        text,
        return_tensors='pt',
        truncation=True,
        max_length=MAX_SEQ_LEN,
    ).to(model.device)
    return model(**enc).logits[0][0].item()


def delta_to_probs(
    score_a: float,
    score_b: float,
    temperature: float,
    tie_weight: float,
) -> np.ndarray:
    """Convert scalar score delta to [p_a, p_b, p_tie] probability vector."""
    diff = score_a - score_b
    p_a  = torch.sigmoid(torch.tensor(diff  * temperature)).item()
    p_b  = torch.sigmoid(torch.tensor(-diff * temperature)).item()
    tie  = (1.0 - abs(p_a - p_b)) * tie_weight
    total = p_a + p_b + tie
    return np.array([p_a / total, p_b / total, tie / total], dtype=np.float32)


print('Score helpers ready')

In [ ]:
def predict_proba(
    df: pd.DataFrame,
    temperature: float,
    tie_weight: float,
    swap: bool = False,
) -> np.ndarray:
    """Score every row; return (N, 3) probability array [p_a, p_b, p_tie]."""
    probs = []
    for i, row in enumerate(df.itertuples(index=False), 1):
        prompt = getattr(row, 'prompt', '')
        ra = getattr(row, 'response_a', '')
        rb = getattr(row, 'response_b', '')
        if swap:
            ra, rb = rb, ra
        sa = score_response(prompt, ra)
        sb = score_response(prompt, rb)
        p  = delta_to_probs(sa, sb, temperature, tie_weight)
        probs.append(p)
        if i % 200 == 0:
            print(f'  {i:,}/{len(df):,}')
    return np.vstack(probs)


def predict_proba_tta(
    df: pd.DataFrame,
    temperature: float,
    tie_weight: float,
) -> np.ndarray:
    """Swap-TTA: average forward + swapped (B,A) pass with flipped A/B columns."""
    fwd = predict_proba(df, temperature, tie_weight, swap=False)
    swp = predict_proba(df, temperature, tie_weight, swap=True)
    # Swapped pass: A and B indices are reversed
    return (fwd + swp[:, [1, 0, 2]]) / 2.0


print('Inference helpers ready')

In [ ]:
test  = pd.read_csv(INPUT / 'test.csv')
train = pd.read_csv(INPUT / 'train.csv')
print(f'Test : {test.shape}')
print(f'Train: {train.shape}')

winner_cols = ['winner_model_a', 'winner_model_b', 'winner_tie']
train['label'] = train[winner_cols].values.argmax(axis=1).astype(int)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
splits  = list(skf.split(train, train['label']))
_, val_idx = splits[0]
df_val = train.iloc[val_idx].reset_index(drop=True)
print(f'Fold 0 val: {len(df_val):,} rows')

In [ ]:
# Grid-search temperature x tie_weight on fold 0 val set
temperatures = [0.5, 1.0, 1.5, 2.0, 3.0]
tie_weights  = [0.1, 0.2, 0.3, 0.4, 0.5]

print('Scoring fold 0 val set (forward pass)...')
# Score once — calibration only changes how we convert deltas, not the raw scores
val_scores_a = []
val_scores_b = []
for i, row in enumerate(df_val.itertuples(index=False), 1):
    prompt = getattr(row, 'prompt', '')
    val_scores_a.append(score_response(prompt, getattr(row, 'response_a', '')))
    val_scores_b.append(score_response(prompt, getattr(row, 'response_b', '')))
    if i % 200 == 0:
        print(f'  {i:,}/{len(df_val):,}')

print('Scoring fold 0 val set (swapped pass)...')
val_scores_a_swp = []
val_scores_b_swp = []
for i, row in enumerate(df_val.itertuples(index=False), 1):
    prompt = getattr(row, 'prompt', '')
    # Swapped: score_a_swp is actually score of response_b in the A slot
    val_scores_a_swp.append(score_response(prompt, getattr(row, 'response_b', '')))
    val_scores_b_swp.append(score_response(prompt, getattr(row, 'response_a', '')))
    if i % 200 == 0:
        print(f'  {i:,}/{len(df_val):,}')

val_labels = df_val['label'].values

best_ll   = float('inf')
best_temp = TEMPERATURE
best_tw   = TIE_WEIGHT

for temp in temperatures:
    for tw in tie_weights:
        fwd_probs = np.vstack([
            delta_to_probs(sa, sb, temp, tw)
            for sa, sb in zip(val_scores_a, val_scores_b)
        ])
        swp_probs = np.vstack([
            delta_to_probs(sa, sb, temp, tw)
            for sa, sb in zip(val_scores_a_swp, val_scores_b_swp)
        ])
        # Swapped: flip A/B columns before averaging
        tta_probs = (fwd_probs + swp_probs[:, [1, 0, 2]]) / 2.0
        ll = log_loss(val_labels, tta_probs)
        if ll < best_ll:
            best_ll   = ll
            best_temp = temp
            best_tw   = tw

print(f'\nBest temperature : {best_temp}')
print(f'Best tie_weight  : {best_tw}')
print(f'Fold 0 OOF log loss (swap-TTA): {best_ll:.4f}')
print(f'v0.1 TF-IDF baseline:            1.0404')
print(f'Δ from baseline:                 {best_ll - 1.0404:+.4f}')

In [ ]:
print(f'Test inference (swap-TTA, temp={best_temp}, tie_weight={best_tw})...')
test_probs = predict_proba_tta(test, best_temp, best_tw)
print(f'Done — shape: {test_probs.shape}')

In [ ]:
sub = pd.DataFrame({
    'id':             test['id'],
    'winner_model_a': test_probs[:, 0],
    'winner_model_b': test_probs[:, 1],
    'winner_tie':     test_probs[:, 2],
})
sub.to_csv(OUTPUT / 'submission.csv', index=False)

print(f'submission.csv: {len(sub):,} rows')
print(f'Prob sums (first 5): {test_probs.sum(axis=1)[:5].round(4)}')
print(f'NaN check: {sub.isnull().sum().sum()}')
print(sub.head())